In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import ClassifierChain
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, hamming_loss, f1_score, precision_recall_curve, make_scorer
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix

# ------------- cargar datos -------------
data_train = pd.read_csv('data/processed/train.csv')
data_val   = pd.read_csv('data/processed/validation.csv')
data_test  = pd.read_csv('data/processed/test.csv')

labels = ["TWF", "HDF", "PWF", "OSF", "RNF"]
features = ['Type','Air temperature [K]','Process temperature [K]','Rotational speed [rpm]','Torque [Nm]', 'Tool wear [min]']

X_train = data_train[features].reset_index(drop=True)
Y_train = data_train[labels].reset_index(drop=True).astype(int)
X_val   = data_val[features].reset_index(drop=True)
Y_val   = data_val[labels].reset_index(drop=True).astype(int)
X_test  = data_test[features].reset_index(drop=True)
Y_test  = data_test[labels].reset_index(drop=True).astype(int)

# sanity check shapes
assert X_train.shape[0] == Y_train.shape[0]
assert X_val.shape[0] == Y_val.shape[0]
assert X_test.shape[0] == Y_test.shape[0]

# ------------- preprocesado -------------
num_cols = ['Air temperature [K]','Process temperature [K]','Rotational speed [rpm]','Torque [Nm]', 'Tool wear [min]']
cat_cols = ['Type']

preproc = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse=False), cat_cols)
], remainder='drop')

# ------------- búsqueda de hiperparámetros del RF base -------------
base_rf = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)

# Pipeline que será pasado a RandomizedSearchCV
pipe = Pipeline([
    ('pre', preproc),
    ('rf', base_rf)
])

param_dist = {
    'rf__n_estimators': [100, 200, 400],
    'rf__max_depth': [None, 20, 40],
    'rf__min_samples_leaf': [1, 2, 4],
    'rf__max_features': ['sqrt', 0.5]
}

cv_inner = MultilabelStratifiedKFold(n_splits=4, shuffle=True, random_state=42)

scorer = make_scorer(f1_score, average='micro')

search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=12,
    scoring=scorer,
    cv=cv_inner,
    n_jobs=-1,
    random_state=42,
    return_train_score=True,
    verbose=1
)

search.fit(X_train, Y_train)

cv_results_rf = pd.DataFrame(search.cv_results_)
cv_results_rf.to_csv('rf_random_search_results.csv', index=False)
joblib.dump(search, 'rf_random_search_object.joblib')

best_params = {k.replace('rf__',''): v for k, v in search.best_params_.items()}
print("Mejores params RF:", best_params)

n_chains = 7
chains = []
for i in range(n_chains):
    rf_clone = RandomForestClassifier(**best_params, class_weight='balanced', random_state=42 + i, n_jobs=-1)
    chain_pipeline = Pipeline([
        ('pre', preproc),
        ('clf', ClassifierChain(rf_clone, order='random', random_state=42 + i))
    ])
    chain_pipeline.fit(X_train, Y_train)
    chains.append(chain_pipeline)
    joblib.dump(chain_pipeline, f'classifier_chain_{i}.joblib')

probs_val = np.zeros((X_val.shape[0], len(labels)))
for chain in chains:
    try:
        p_list = chain.predict_proba(X_val) 
        p_mat = np.vstack([p[:,1] for p in p_list]).T
    except Exception:
        p_mat = chain.predict(X_val).astype(float)
    probs_val += p_mat

probs_val /= n_chains

# Optimizar umbrales por etiqueta maximizando F1 en validation
best_thresholds = []
for i, label in enumerate(labels):
    y_true = Y_val.iloc[:, i].values
    probs = probs_val[:, i]
    if len(np.unique(y_true)) == 1:
        best_thresholds.append(0.5)
        continue
    precisions, recalls, thresholds = precision_recall_curve(y_true, probs)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-12)
    # elegir el umbral que maximiza F1
    best_idx = np.nanargmax(f1_scores)
    if best_idx == 0:
        t = 0.5 if len(thresholds) == 0 else thresholds[0]
    else:
        t = thresholds[best_idx-1] if best_idx-1 < len(thresholds) else 0.5
    best_thresholds.append(float(t))

pd.Series(best_thresholds, index=labels).to_csv('best_thresholds.csv')

probs_test = np.zeros((X_test.shape[0], len(labels)))
for chain in chains:
    try:
        p_list = chain.predict_proba(X_test)
        p_mat = np.vstack([p[:,1] for p in p_list]).T
    except Exception:
        p_mat = chain.predict(X_test).astype(float)
    probs_test += p_mat

probs_test /= n_chains

y_pred_test = (probs_test >= np.array(best_thresholds)).astype(int)
y_pred_test_df = pd.DataFrame(y_pred_test, columns=labels, index=X_test.index)

print("Shapes: X_test, Y_test, y_pred_test:", X_test.shape, Y_test.shape, y_pred_test.shape)
print("Positivos reales por etiqueta:\n", Y_test.sum())
print("Positivos predichos por etiqueta:\n", y_pred_test_df.sum())

# Matrices de confusión por etiqueta
for i, col in enumerate(labels):
    tn, fp, fn, tp = confusion_matrix(Y_test.iloc[:, i], y_pred_test_df.iloc[:, i]).ravel()
    print(f"{col}: TN={tn} FP={fp} FN={fn} TP={tp}")

# Reporte final
print(classification_report(Y_test, y_pred_test_df, target_names=labels))
print("Hamming loss:", hamming_loss(Y_test, y_pred_test_df))
print("F1 micro:", f1_score(Y_test, y_pred_test_df, average='micro'))
print("F1 macro:", f1_score(Y_test, y_pred_test_df, average='macro'))

pd.DataFrame(y_pred_test, columns=labels).to_csv('y_pred_test_chains.csv', index=False)
pd.DataFrame(probs_test, columns=[f"{l}_prob" for l in labels]).to_csv('y_prob_test_chains.csv', index=False)

cv_outer = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# OOF con una sola cadena para diagnóstico
chain_oof = Pipeline([
    ('pre', preproc),
    ('clf', ClassifierChain(RandomForestClassifier(**best_params, class_weight='balanced', random_state=0, n_jobs=-1), order='random', random_state=0))
])
y_oof = cross_val_predict(chain_oof, X_train, Y_train, cv=cv_outer, method='predict', n_jobs=-1)
pd.DataFrame(y_oof, columns=labels).to_csv('oof_preds_chain.csv', index=False)

print("OOF F1 micro:", f1_score(Y_train, y_oof, average='micro'))
print("OOF F1 macro:", f1_score(Y_train, y_oof, average='macro'))


TypeError: OneHotEncoder.__init__() got an unexpected keyword argument 'sparse'